## Backstage

[Backstage.io](https://backstage.io) ist eine Open-Source-Developer-Plattform, die von Spotify entwickelt wurde, um die Verwaltung von Softwareprojekten zu vereinfachen. 

Mit Backstage können Unternehmen eine zentrale Anlaufstelle für all ihre internen Tools, Services, Dokumentationen und Software-Komponenten schaffen. 

Im Mittelpunkt steht das Konzept des "Service Catalogs", der alle Applikationen und Services übersichtlich darstellt. 

Durch Plugins lässt sich Backstage flexibel erweitern und an individuelle Bedürfnisse anpassen. Ziel ist es, Entwickler:innen die tägliche Arbeit zu erleichtern und eine einheitliche Nutzererfahrung über verschiedene Tools hinweg zu bieten.


### Installation

Dazu greifen wir auf Scripts aus dem [Lern Cloud Projekt](https://github.com/mc-b/lerncloud/tree/main/services) zurück:

In [ ]:
%%bash
curl -sfL https://raw.githubusercontent.com/mc-b/lerncloud/refs/heads/main/services/backstage.sh | bash -

Nach der Installation der benötigten Umgebung (node.js etc.) holen wir uns eine fertig Installierte Umgebung. 

Dieses ist aus Gründen der Einfachheit in ein Container Image verpackt. Der Container selber hat keine Funktion.

In [ ]:
%%bash
cd
docker run --rm registry.gitlab.com/ch-mc-b/autoshop-ms/infra/backstage/backstage-app:0.0.1 /bin/cat /app/backstage.tgz | tar xzvf -

### Backstage UI

Nach dem Ausführen der untenstehenden Schritte ist das Backstage UI unter folgendem URL erreichbar:

In [ ]:
%%bash
echo "http://$(cat ~/work/server-ip):3000"

### Einrichten der Authentifizierung

Für Backstage stehen Ihnen verschiedene Authentifizierungsanbieter zur Verfügung. Hier verwenden wir GitHub.

**Fügt in GitHub eine neue App hinzu**

Geht zu [https://github.com/settings/applications/new](https://github.com/settings/applications/new), um Eure OAuth-App zu erstellen.


In [ ]:
%%bash
echo "Backstage-Frontend        : http://$(cat ~/work/server-ip):3000"
echo "Authorization callback URL: http://$(cat ~/work/server-ip):7007/api/auth/github/handler/frame"

### Einrichten Personal access tokens (classic)

Um neue Repositories erstellen zu können, brauchen wir einen Personal access tokens (classic).

Geht zu [https://github.com/settings/tokens](https://github.com/settings/tokens) und erstellt einen Classic Token.

Dieser braucht mindestens folgende Rechte:

* Reading software components:
    * repo
* Reading organization data:
    * read:org
    * read:user
    * user:email
* Publishing software templates:
    * repo
    * workflow (if templates include GitHub workflows)  


### Setzen der Umgebung

Setzt die erstellen Token etc. als Umgebungsvariablen

In [ ]:
import os
os.environ['BACKSTAGE_ORG']='Auto Shop Group'
os.environ['GITHUB_CLIENT_ID']=''
os.environ['GITHUB_SECRET']=''
os.environ['GITHUB_TOKEN']=''


### Startet Backstage

Die Umgebung kann mittels des **Stopp** Buttons wieder bendet werden.

Es wird eine In-Memory Datenbank verwendet.

In [ ]:
%%bash
cd ~/backstage
. ~/.nvm/nvm.sh 
export NODE_OPTIONS=--no-node-snapshot
export BACKSTAGE_HOST="$(cat ~/work/server-ip)"
yarn start


---
### Backstage als Service einrichten



In [ ]:
mkdir -p ~/.config/systemd/user
cat <<EOF > ~/.config/systemd/user/backstage.service
[Unit]
Description=Backstage.io
After=network.target

[Service]
Environment="NODE_OPTIONS=--no-node-snapshot"
Environment="BACKSTAGE_ORG=${BACKSTAGE_ORG}"
Environment="BACKSTAGE_HOST=$(cat ~/work/server-ip)"
Environment="GITHUB_CLIENT_ID=${GITHUB_CLIENT_ID}"
Environment="GITHUB_SECRET=${GITHUB_SECRET}"
Environment="GITHUB_TOKEN=${GITHUB_TOKEN}"
Type=simple
WorkingDirectory=/home/ubuntu/backstage
ExecStartPre=/bin/bash -c 'source /home/ubuntu/.nvm/nvm.sh'
ExecStart=/home/ubuntu/.nvm/versions/node/v20.19.1/bin/node /home/ubuntu/.nvm/versions/node/v20.19.1/bin/yarn start
Restart=on-failure

[Install]
WantedBy=default.target
EOF
cat ~/.config/systemd/user/backstage.service

**Backstage Service starten**

    systemctl --user daemon-reload
    systemctl --user  start backstage


**Status überprüfen**

    systemctl --user status backstage.service